# 🏎️ F1 Pit Stop Prediction

**Kaggle Playground Series S6E5** — Binary classification: predict whether a driver will pit on the next lap.

## Approach
- Feature engineering on tyre life, race progress, and lap-time trends
- XGBoost with 5-fold Stratified Cross-Validation (Out-of-Fold predictions)
- Label encoding fitted **only on training data** to avoid data leakage
- Evaluation metric: ROC-AUC

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1. Data Loading & Preprocessing

In [ ]:
def load_and_preprocess():
    """Load train/test CSVs and cast target column to int."""
    train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
    test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/test.csv")

    # Cast target to integer (0 / 1)
    train['PitNextLap'] = train['PitNextLap'].astype(int)

    print(f"Train shape : {train.shape}")
    print(f"Test shape  : {test.shape}")
    print(f"Pit rate    : {train['PitNextLap'].mean():.4f}")
    return train, test

train, test = load_and_preprocess()
train.head()

## 2. Feature Engineering

> **Important — no data leakage:** `LabelEncoder` is fitted **only on training data** and then applied to the test set using the same mapping.

In [ ]:
CAT_COLS = ['Driver', 'Compound', 'Race']

def feature_engineering(train_df, test_df):
    """
    Create new features and encode categoricals.
    LabelEncoders are fit on train only, then applied to test — prevents leakage.
    """
    for df in [train_df, test_df]:
        # Tyre strategy features
        df['TyreUseRatio']     = df['TyreLife'] / (df['LapNumber'] + 1)
        df['RemainingRace']    = 1 - df['RaceProgress']

        # Degradation features
        df['Degradation_Rate'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)
        df['Is_Late_Race']     = (df['RaceProgress'] > 0.8).astype(int)

    # Lap-time average per driver — computed on train, mapped to test
    driver_avg = train_df.groupby('Driver')['LapTime (s)'].mean().to_dict()
    for df in [train_df, test_df]:
        df['Avg_LapTime_Driver'] = df['Driver'].map(driver_avg)

    # Label encoding: fit on train, transform both
    encoders = {}
    for col in CAT_COLS:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        # Handle unseen categories in test gracefully
        test_df[col]  = test_df[col].astype(str).map(
            lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1
        )
        encoders[col] = le

    return train_df, test_df, encoders

train, test, encoders = feature_engineering(train, test)
print("Features created. Train columns:", list(train.columns))

## 3. Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Pit rate by tyre life
train.groupby('TyreLife')['PitNextLap'].mean().plot(
    ax=axes[0], title='Pit Rate by Tyre Life', xlabel='TyreLife', ylabel='Pit Probability'
)

# Pit rate by race progress bucket
train['RaceProgressBucket'] = pd.cut(train['RaceProgress'], bins=10)
train.groupby('RaceProgressBucket')['PitNextLap'].mean().plot(
    kind='bar', ax=axes[1], title='Pit Rate by Race Progress', rot=45
)

plt.tight_layout()
plt.show()

## 4. Model Parameters

XGBoost parameters — tuned with Optuna as a starting point.

In [ ]:
XGB_PARAMS = {
    'n_estimators'      : 2000,
    'learning_rate'     : 0.025,
    'max_depth'         : 6,
    'subsample'         : 0.8,
    'colsample_bytree'  : 0.8,
    'n_jobs'            : -1,
    'eval_metric'       : 'logloss',
    'objective'         : 'binary:logistic',
    'random_state'      : 42,
    'tree_method'       : 'hist',   # fast histogram-based training
    'early_stopping_rounds': 100,
}

## 5. Training — Stratified 5-Fold Cross-Validation (OOF)

In [ ]:
def train_and_validate(train_df, test_df):
    """
    Stratified K-Fold training with Out-of-Fold predictions.
    Returns averaged test predictions.
    """
    drop_cols = ['id', 'PitNextLap', 'PitStop', 'RaceProgressBucket']
    features  = [c for c in train_df.columns if c not in drop_cols]

    X      = train_df[features]
    y      = train_df['PitNextLap']
    X_test = test_df[features]

    skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))

    print(f"Training XGBoost | features: {len(features)} | folds: 5\n")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = xgb.XGBClassifier(**XGB_PARAMS)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        fold_preds          = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx]  = fold_preds
        test_preds         += model.predict_proba(X_test)[:, 1] / 5

        score = roc_auc_score(y_val, fold_preds)
        print(f"  Fold {fold+1}  AUC: {score:.5f}  |  best iteration: {model.best_iteration}")

    oof_auc = roc_auc_score(y, oof_preds)
    print(f"\n  Overall OOF AUC: {oof_auc:.5f}")
    return test_preds, model   # return last fold model for feature importance

final_predictions, last_model = train_and_validate(train, test)

## 6. Feature Importance

In [ ]:
drop_cols = ['id', 'PitNextLap', 'PitStop', 'RaceProgressBucket']
features  = [c for c in train.columns if c not in drop_cols]

fi = pd.Series(last_model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
fi.head(20).plot(kind='barh')
plt.title('Top 20 Feature Importances (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Submission

In [ ]:
submission = pd.DataFrame({
    'id'         : test['id'],
    'PitNextLap' : final_predictions
})
submission.to_csv('submission.csv', index=False)
print("Submission saved → submission.csv")
submission.head()